# 📰 AI Research & News Summarizer Agent

A portfolio-ready AI engineering notebook for collecting research/news content, extracting useful information, generating concise summaries, comparing sources, and producing a grounded research brief.

**Core stack:** Python + NLP + optional LLM + agent workflow concepts.


## 🎯 Project Goals

- Collect and normalize article/research content.
- Extract titles, sources, dates, and article text.
- Rank content by relevance.
- Generate concise summaries with an extractive baseline.
- Combine multiple articles into a research brief.
- Identify common themes and conflicting claims.
- Add an optional LLM-powered summarization layer.
- Design a production-ready agent workflow with source grounding.


In [ ]:
import re
import math
import numpy as np
import pandas as pd
from collections import Counter

print("Environment ready.")


## 1. 🗂️ Create a Sample Research/News Corpus

The notebook uses a small local corpus so the core workflow runs without external APIs. Replace this data with RSS feeds, News APIs, web extraction, or research-paper sources in production.


In [ ]:
documents = [
    {
        "title": "AI Agents Move Toward Tool-Based Workflows",
        "source": "Tech Research Daily",
        "date": "2026-09-28",
        "topic": "AI Agents",
        "text": (
            "AI agent systems are increasingly being designed around tools, structured workflows, "
            "memory, and explicit evaluation. Researchers and engineering teams are focusing on "
            "reliability, observability, permissions, and human approval for high-impact actions."
        )
    },
    {
        "title": "Retrieval-Augmented Generation Improves Enterprise Search",
        "source": "Enterprise AI Review",
        "date": "2026-09-27",
        "topic": "RAG",
        "text": (
            "Retrieval-augmented generation combines information retrieval with language models. "
            "Recent enterprise systems emphasize document chunking, metadata filters, hybrid search, "
            "reranking, citations, and evaluation to reduce unsupported answers."
        )
    },
    {
        "title": "Small Language Models Gain Attention for Private AI",
        "source": "ML Systems Journal",
        "date": "2026-09-26",
        "topic": "LLMs",
        "text": (
            "Smaller language models are attractive for lower-cost inference, edge deployment, and "
            "privacy-sensitive applications. Model selection increasingly considers latency, accuracy, "
            "hardware requirements, and the ability to specialize models for narrow tasks."
        )
    },
    {
        "title": "AI Evaluation Becomes a Core Engineering Discipline",
        "source": "Applied ML Weekly",
        "date": "2026-09-25",
        "topic": "AI Evaluation",
        "text": (
            "Teams building AI applications are investing more heavily in evaluation datasets, "
            "automated quality checks, traceability, and production monitoring. Evaluation is moving "
            "beyond model accuracy toward groundedness, relevance, safety, latency, and cost."
        )
    }
]

df = pd.DataFrame(documents)
df


## 2. 🔎 Keyword-Based Research Retrieval

This lightweight retrieval function provides a local baseline. In a production system it can be replaced or combined with BM25, vector embeddings, hybrid retrieval, or a web/news search API.


In [ ]:
def tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

def search_documents(query, data, top_k=3):
    query_terms = set(tokenize(query))
    scored = []
    for _, row in data.iterrows():
        text_terms = tokenize(row["title"] + " " + row["text"])
        counts = Counter(text_terms)
        score = sum(counts[t] for t in query_terms)
        scored.append((score, row["title"], row["source"]))
    scored.sort(reverse=True)
    return pd.DataFrame(scored[:top_k], columns=["score", "title", "source"])

search_documents("AI agents tools reliability", df)


## 3. ✂️ Extractive Summarization Baseline

This baseline ranks sentences using keyword frequency. It is intentionally simple and transparent, making it a useful benchmark before adding an LLM.


In [ ]:
STOPWORDS = {
    "the","a","an","and","or","of","to","in","on","for","with","is","are",
    "as","by","from","that","this","it","be","more","into","than","their"
}

def sentence_split(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def extractive_summary(text, max_sentences=2):
    sentences = sentence_split(text)
    words = [w for w in tokenize(text) if w not in STOPWORDS]
    freq = Counter(words)
    scored = []
    for i, sentence in enumerate(sentences):
        tokens = [w for w in tokenize(sentence) if w not in STOPWORDS]
        score = sum(freq[w] for w in tokens) / max(1, len(tokens))
        scored.append((score, i, sentence))
    selected = sorted(scored, reverse=True)[:max_sentences]
    selected = sorted(selected, key=lambda x: x[1])
    return " ".join(x[2] for x in selected)

df["summary"] = df["text"].apply(extractive_summary)
df[["title", "summary"]]


## 4. 📊 Build a Research Brief

A useful research assistant should not only summarize individual sources. It should synthesize the retrieved evidence into a compact briefing.


In [ ]:
def build_research_brief(topic, data):
    relevant = data[data["topic"].str.lower().str.contains(topic.lower(), na=False)]
    if relevant.empty:
        relevant = data
    rows = []
    for _, row in relevant.iterrows():
        rows.append({
            "source": row["source"],
            "title": row["title"],
            "date": row["date"],
            "summary": row["summary"]
        })
    return pd.DataFrame(rows)

brief = build_research_brief("AI", df)
brief


## 5. 🧠 Theme Extraction

Theme counts can help an agent identify which concepts appear repeatedly across a research set.


In [ ]:
def common_terms(data, top_n=15):
    text = " ".join(data["title"].tolist() + data["text"].tolist())
    terms = [
        w for w in tokenize(text)
        if w not in STOPWORDS and len(w) > 3
    ]
    return Counter(terms).most_common(top_n)

common_terms(df)


## 6. ⚖️ Source Comparison

For real research workflows, multiple sources should be retained rather than collapsing everything into one unsupported answer. This table makes the evidence trail visible.


In [ ]:
comparison = df[[
    "title", "source", "date", "topic", "summary"
]].sort_values("date", ascending=False)

comparison


## 7. 🤖 Optional LLM Summarization Layer

The deterministic baseline above can be upgraded with an OpenAI-compatible LLM such as an OpenRouter model.

Recommended design principles:

1. Retrieve relevant sources first.
2. Pass the source text and metadata to the model.
3. Instruct the model to use only supplied evidence.
4. Require source references in the output.
5. Separate facts from interpretation.
6. Never treat an LLM-generated statement as a verified fact without source support.


In [ ]:
# Optional example using an OpenAI-compatible API.
# Install first if needed:
# pip install openai

# import os
# from openai import OpenAI
#
# client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.environ["OPENROUTER_API_KEY"]
# )
#
# source_text = "\n\n".join(
#     f"Source: {r.source}\nTitle: {r.title}\nText: {r.text}"
#     for r in df.itertuples()
# )
#
# prompt = f'''
# Summarize the supplied research/news sources.
# Use ONLY the provided evidence.
# Clearly separate:
# 1. Key findings
# 2. Important differences
# 3. Emerging themes
# 4. Open questions
# Cite each claim using the supplied source title.
#
# SOURCES:
# {source_text}
# '''
#
# response = client.chat.completions.create(
#     model="openai/gpt-4.1-mini",
#     messages=[{"role": "user", "content": prompt}]
# )
# print(response.choices[0].message.content)


## 8. 🧩 Agent Workflow Design

A production Research & News Summarizer Agent can be organized as:

```text
User Research Question
        ↓
Query Planner
        ↓
News / Web / Research Retrieval
        ↓
Document Cleaner
        ↓
Relevance Ranker
        ↓
Summarization Agent
        ↓
Cross-Source Synthesis
        ↓
Fact / Claim Grounding
        ↓
Research Brief
        ↓
Human Review
```

Possible frameworks:

- LangGraph for stateful agent orchestration
- LangChain for retrieval and model integrations
- MCP for connecting approved external tools
- Vector databases for semantic retrieval
- OpenRouter/OpenAI-compatible APIs for LLM inference


## 9. 🛡️ Reliability & Responsible AI

A research summarizer should minimize hallucination and source distortion.

Recommended safeguards:

- 🔗 Preserve source URLs and metadata.
- 📌 Cite claims back to sources.
- 🧠 Distinguish source facts from model interpretation.
- ⚠️ Flag conflicting sources.
- 🕒 Display publication dates.
- 🔄 Refresh fast-changing topics before producing a final brief.
- 👤 Keep human review for high-impact research conclusions.
- 🔒 Avoid sending confidential documents to external models without authorization.


## 10. 🚀 Production Roadmap

### Phase 1 — Local Prototype
- [x] Document ingestion
- [x] Keyword retrieval
- [x] Extractive summarization
- [x] Research brief generation

### Phase 2 — Intelligent Retrieval
- [ ] RSS/news API integration
- [ ] Web search integration
- [ ] Sentence-transformer embeddings
- [ ] Vector database
- [ ] Hybrid retrieval
- [ ] Reranking

### Phase 3 — Generative AI
- [ ] LLM summarization
- [ ] Citation-aware prompts
- [ ] Structured JSON outputs
- [ ] Claim verification
- [ ] Multi-document synthesis

### Phase 4 — Agentic System
- [ ] LangGraph workflow
- [ ] Research planner agent
- [ ] Retrieval agent
- [ ] Summarization agent
- [ ] Fact-checking agent
- [ ] Report-generation agent
- [ ] Human approval

### Phase 5 — Production
- [ ] FastAPI backend
- [ ] React/Next.js dashboard
- [ ] Scheduled research jobs
- [ ] Persistent research history
- [ ] Observability and evaluation
- [ ] Cost and latency monitoring


## ✅ Conclusion

This notebook demonstrates the foundation of an **AI Research & News Summarizer Agent**: retrieval, relevance ranking, source-aware summarization, cross-document synthesis, and an architecture that can evolve into a multi-agent research system.

The key engineering principle is simple:

> **Retrieve first, ground the answer in evidence, then summarize.**
